<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 컴퓨터 비전 · 03. GoogLeNet / Inception

## Going Deeper with Convolutions

- **원 논문:** [Going Deeper with Convolutions](https://arxiv.org/abs/1409.4842)
- **저자 / 발표:** Christian Szegedy et al. · CVPR 2015 (2014)
- **난이도 / 예상 시간:** 중급 · 약 120분
- **선수 지식:** Python, PyTorch tensor, 합성곱, 역전파의 기초
- **로컬 학습 데이터:** `data/field_curriculum/vision_shapes.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** multi-scale 병렬 분기와 1×1 dimension reduction을 재현한다.

**축소하거나 생략한 부분:** 22-layer GoogLeNet과 auxiliary head 대신 Inception module 두 개를 사용한다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §4 Fig. 2(a): naïve Inception | Task 2 branches | concat shape |
| §4 Fig. 2(b): dimension reduction | Task 1 | parameter 절감 |
| §5 Table 1: GoogLeNet | Task 2 | 두 modules |
| §6.3: classification | Task 3 | loss/accuracy |

## 핵심 아이디어와 수식

$$y=\operatorname{Concat}\!\left(f_{1\times1}(x),f_{3\times3}(x),f_{5\times5}(x),f_p(x)\right)$$

**기호 해설:** 각 f는 서로 다른 공간 척도의 branch이고 Concat은 channel 축 결합이다.

**코드 대응:** Task 1은 1×1 reduction 비용을 계산하고 Task 2가 네 branch를 각각 구현한다.

**입력과 출력 shape:** 입력 [B,C,H,W] → 네 branch [B,4/6/3/3,H,W] → [B,16,H,W].

**포트폴리오 구현 범위:** 원 논문의 전체 stage와 auxiliary classifier는 제외하고 module 내부 계산은 보존한다.

아래 코드는 결과만 내는 라이브러리 호출로 핵심을 감추지 않는다. config, 논문 고유
연산, 모델의 `forward`, 목적함수, `train_step`, 평가 함수를 순서대로 직접 작성한다.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import itertools
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(20260814)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device


def locate(relative_path):
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / relative_path
        if candidate.exists():
            return candidate
    raise FileNotFoundError(relative_path)


dataset_path = locate("data/field_curriculum/vision_shapes.npz")
raw = np.load(dataset_path)
images = torch.from_numpy(raw["images"]).float()
labels = torch.from_numpy(raw["labels"]).long()
masks = torch.from_numpy(raw["masks"]).float()
boxes = torch.from_numpy(raw["boxes"]).float()
train_idx = torch.from_numpy(raw["train_idx"][:128]).long()
test_idx = torch.from_numpy(raw["test_idx"][:64]).long()
images, labels, masks, boxes = ACCELERATOR.move(
    images,
    labels,
    masks,
    boxes,
)
train_idx, test_idx = ACCELERATOR.move(train_idx, test_idx)
class_names = raw["class_names"].tolist()
x_train = images[train_idx]
y_train = labels[train_idx]
assert images.shape == (480, 1, 16, 16)
assert masks.shape == (480, 16, 16)
assert boxes.shape == (480, 4)
print(ACCELERATOR.summary())
print(dataset_path.name, images.shape, class_names)

### 구현 단계 1 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §4 Fig. 2(a): naïve Inception
- **노트북에서 구현할 부분:** Task 2 branches
- **구현 이유:** 논문의 계산 경로를 작은 tensor로 보존하면서 각 중간값을
  assertion으로 관찰하기 위해서다.
- **완료 증거:** concat shape

먼저 입력 shape와 기대 출력 shape를 종이에 적은 뒤 아래 TODO를 구현한다.

In [ ]:
@dataclass
class InceptionConfig:
    stem_channels: int = 8
    branch_channels: tuple = (4, 6, 3, 3)
    num_classes: int = 4
    learning_rate: float = 0.012
    steps: int = 12


def branch_parameter_count(in_channels, reduced_channels, out_channels, kernel):
    reduction = in_channels * reduced_channels
    spatial = reduced_channels * out_channels * kernel * kernel
    return reduction + spatial


config = InceptionConfig()
direct = 32 * 16 * 5 * 5
reduced = branch_parameter_count(32, 4, 16, 5)
assert reduced < direct
assert reduced / direct < 0.2

### 구현 단계 2 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §4 Fig. 2(b): dimension reduction
- **노트북에서 구현할 부분:** Task 1
- **구현 이유:** 논문의 계산 경로를 작은 tensor로 보존하면서 각 중간값을
  assertion으로 관찰하기 위해서다.
- **완료 증거:** parameter 절감

먼저 입력 shape와 기대 출력 shape를 종이에 적은 뒤 아래 TODO를 구현한다.

In [ ]:
class InceptionBlock(nn.Module):
    def __init__(self, in_channels, config):
        super().__init__()
        c1, c3, c5, cp = config.branch_channels
        self.branch1 = nn.Conv2d(in_channels, c1, 1)
        self.reduce3 = nn.Conv2d(in_channels, 4, 1)
        self.branch3 = nn.Conv2d(4, c3, 3, padding=1)
        self.reduce5 = nn.Conv2d(in_channels, 2, 1)
        self.branch5 = nn.Conv2d(2, c5, 5, padding=2)
        self.pool_projection = nn.Conv2d(in_channels, cp, 1)

    def forward(self, inputs):
        branch1 = F.relu(self.branch1(inputs))
        branch3 = F.relu(self.reduce3(inputs))
        branch3 = F.relu(self.branch3(branch3))
        branch5 = F.relu(self.reduce5(inputs))
        branch5 = F.relu(self.branch5(branch5))
        pooled = F.max_pool2d(inputs, 3, stride=1, padding=1)
        pooled = F.relu(self.pool_projection(pooled))
        return torch.cat((branch1, branch3, branch5, pooled), dim=1)


class InceptionMini(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.stem = nn.Conv2d(1, config.stem_channels, 3, padding=1)
        self.block1 = InceptionBlock(config.stem_channels, config)
        self.block2 = InceptionBlock(sum(config.branch_channels), config)
        self.classifier = nn.Linear(16, config.num_classes)

    def forward(self, inputs):
        features = F.relu(self.stem(inputs))
        features = self.block1(features)
        features = self.block2(features)
        return self.classifier(features.mean(dim=(2, 3)))


model = InceptionMini(config).to(DEVICE)
assert model.block1(images.new_zeros(2, 8, 16, 16)).shape == (2, 16, 16, 16)
assert model(images[:2]).shape == (2, 4)

### 구현 단계 3 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §6.3: classification
- **노트북에서 구현할 부분:** Task 3
- **구현 이유:** 논문의 계산 경로를 작은 tensor로 보존하면서 각 중간값을
  assertion으로 관찰하기 위해서다.
- **완료 증거:** loss/accuracy

먼저 입력 shape와 기대 출력 shape를 종이에 적은 뒤 아래 TODO를 구현한다.

In [ ]:
def classification_loss(logits, targets):
    return F.cross_entropy(logits, targets)


def train_step(model, optimizer, inputs, targets):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    logits = model(inputs)
    loss = classification_loss(logits, targets)
    loss.backward()
    optimizer.step()
    return float(loss.detach())


def evaluate_classifier(model, inputs, targets):
    model.eval()
    with torch.no_grad():
        predictions = model(inputs).argmax(dim=1)
    return float((predictions == targets).float().mean())


optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
losses = []
for step in range(config.steps):
    step_loss = train_step(model, optimizer, x_train, y_train)
    losses.append(step_loss)

accuracy = evaluate_classifier(
    model,
    images[test_idx],
    labels[test_idx],
)
assert min(losses[3:]) < losses[0]
assert 0.0 <= accuracy <= 1.0
plt.figure(figsize=(5, 3))
plt.plot(losses)
plt.xlabel("step")
plt.ylabel("cross entropy")
plt.title("Inception mini: accuracy={:.1%}".format(accuracy))
plt.tight_layout()
plt.show()
print(f"loss={losses[0]:.3f} -> {losses[-1]:.3f}")

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> 22-layer GoogLeNet과 auxiliary head 대신 Inception module 두 개를 사용한다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.